#  Assignment 2: Strands Travel Planner

Build an AI agent using **Strands Agents** that creates a **one-day travel plan** for a city.

The agent must:
1. Use a **weather tool** to get the current weather.
2. Use a **web/search tool** to find **3 popular attractions**.
3. Use a **calculator tool** to estimate the total cost of visiting the attractions.
4. Combine the results into a concise **one-day itinerary**.

### How this notebook works

```
City ──▶ Strands Agent (Groq · openai/gpt-oss-120b)
            │  decides which tool to call next (SequentialToolExecutor = one at a time)
            ├─▶ get_current_weather(city)      → OpenWeather API
            ├─▶ search_web(query)              → Tavily web search
            ├─▶ calculate_total_cost(costs)    → plain Python addition
            ▼
      One-day itinerary
```

| Rubric item | Where |
|---|---|
| Strands agent setup and configuration | Sections 1 and 5 |
| Correct use of weather and search tools | Sections 2 and 3 |
| Correct use of calculator/tool calling | Section 4 |
| Accurate itinerary and cost calculation | Sections 6 and 7 |
| Code quality and clear final output | Throughout |

Same setup as the instructor's `GenAI02_strands_groq.ipynb`: Groq is used through Strands' `OpenAIModel`, tools are plain Python functions with `@tool`, and search uses Tavily.

## 1. Imports and configuration

Keys are read from the project's `.env` file with `load_dotenv()` — they are never printed.
This notebook needs `GROQ_API_KEY`, `OPENWEATHER_API_KEY` and `TAVILY_API_KEY`.

In [21]:
import os

import requests
from dotenv import load_dotenv

# Strands core imports (same as the instructor notebook)
from strands import Agent, tool
from strands.models.openai import OpenAIModel
from strands.tools.executors import SequentialToolExecutor
from tavily import TavilyClient

# Load environment variables (.env)
load_dotenv()

# Check the keys exist — print only "found"/"missing", never the values.
REQUIRED_KEYS = ["GROQ_API_KEY", "OPENWEATHER_API_KEY", "TAVILY_API_KEY"]
missing_keys = [name for name in REQUIRED_KEYS if not os.getenv(name)]
for name in REQUIRED_KEYS:
    print(f"{name}: {'missing ✗' if name in missing_keys else 'found ✓'}")
if missing_keys:
    raise RuntimeError(f"Add these to your .env file, then restart the kernel: {', '.join(missing_keys)}")

# Groq model inside Strands — Groq offers an OpenAI-compatible API, so we use
# Strands' OpenAIModel with Groq's base_url (exactly as in the instructor notebook).
groq_model = OpenAIModel(
    model_id="openai/gpt-oss-120b",
    client_args={
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1",
    },
)

OPENWEATHER_URL = "https://api.openweathermap.org/data/2.5/weather"
REQUEST_TIMEOUT_SECONDS = 10

# A simple list where every tool records that it was called,
# so we can show which tools the agent actually used (Section 7).
tool_calls_log = []

print("Groq model ready: openai/gpt-oss-120b")

GROQ_API_KEY: found ✓
OPENWEATHER_API_KEY: found ✓
TAVILY_API_KEY: found ✓
Groq model ready: openai/gpt-oss-120b


## 2. Weather tool

Same idea as my Assignment 1 weather function: call OpenWeather, return a small dict,
and **return an error message instead of crashing** if something goes wrong.
The docstring and type hints tell the model what the tool does and what to pass in.

In [22]:
@tool
def get_current_weather(city: str) -> dict:
    """Gets the current weather for a city from the OpenWeather API (metric units, °C).

    Args:
        city: The city name, optionally with a country code, e.g. "Kathmandu" or "Kathmandu,NP".
    """
    print(f"\n[TOOL EXECUTED] get_current_weather(city='{city}')")
    tool_calls_log.append(("Weather tool", f"get_current_weather('{city}')"))

    def failure(message: str) -> dict:
        print(f"    → ERROR: {message}")
        return {"ok": False, "city": city, "error": message}

    params = {"q": city, "appid": os.getenv("OPENWEATHER_API_KEY"), "units": "metric"}

    try:
        response = requests.get(OPENWEATHER_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    except requests.exceptions.Timeout:
        return failure("OpenWeather did not respond in time.")
    except requests.exceptions.RequestException:
        return failure("Could not connect to OpenWeather.")

    if response.status_code == 401:
        return failure("OpenWeather rejected the API key (HTTP 401).")
    if response.status_code == 404:
        return failure(f"City not found by OpenWeather: '{city}'.")
    if response.status_code != 200:
        return failure(f"OpenWeather returned HTTP {response.status_code}.")

    try:
        data = response.json()
        result = {
            "ok": True,
            "city": data["name"],
            "country": data.get("sys", {}).get("country", ""),
            "temperature_c": round(float(data["main"]["temp"]), 1),
            "feels_like_c": round(float(data["main"]["feels_like"]), 1),
            "humidity_percent": data["main"].get("humidity"),
            "description": data["weather"][0]["description"],
        }
    except (ValueError, KeyError, IndexError, TypeError):
        return failure("OpenWeather's response was missing expected fields.")

    print(f"    → {result['city']}, {result['country']}: {result['temperature_c']} °C, {result['description']}")
    return result

## 3. Search tool (Tavily)

Same Tavily approach as the instructor's `search_web` tool. The agent writes its own search
queries — for example *"top tourist attractions in Kathmandu"* and *"Boudhanath Stupa entry fee"* —
so the attractions and prices come from the web, not from this code.

Tavily results can be long (the instructor notes it is *token intensive*), so each result is
shortened before it is sent back to the model.

In [23]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

MAX_SEARCH_RESULTS = 3
MAX_CONTENT_CHARS = 600   # keep results short to save tokens


@tool
def search_web(query: str) -> dict:
    """Performs a web search using Tavily to get real-time information,
    such as popular attractions in a city or an attraction's entry fee.

    Args:
        query: The search query string, e.g. "top 3 tourist attractions in Kathmandu".
    """
    print(f"\n[TOOL EXECUTED] search_web(query='{query}')")
    tool_calls_log.append(("Search tool", f"search_web('{query}')"))

    try:
        response = tavily_client.search(query=query, search_depth="basic", max_results=MAX_SEARCH_RESULTS)
    except Exception as exc:  # e.g. network problem, bad key, usage limit
        message = f"Web search failed ({type(exc).__name__})."
        print(f"    → ERROR: {message}")
        return {"ok": False, "query": query, "error": message}

    results = [
        {
            "title": r.get("title"),
            "content": (r.get("content") or "")[:MAX_CONTENT_CHARS],
            "url": r.get("url"),
        }
        for r in response.get("results", [])
    ]
    print(f"    → {len(results)} result(s): " + " | ".join((r["title"] or "")[:50] for r in results))
    return {"ok": True, "query": query, "results": results}

## 4. Calculator tool

The **total cost is calculated here in Python**, not guessed by the model.
The agent passes in the admission costs it found with the search tool, and this tool adds them up.

- All costs must be in **one currency** (the agent is told to convert or keep them consistent).
- A free attraction counts as `0`.

In [24]:
last_calculation = {}

NUMBER_OF_ATTRACTIONS = 3


@tool
def calculate_total_cost(attractions: list[str], costs: list[float], currency: str) -> dict:
    """Adds up the admission costs of exactly 3 attractions and returns the exact total.
    Always use this tool for the total — do not add the numbers yourself.

    Args:
        attractions: The names of the 3 attractions, in the same order as `costs`.
        costs: The admission cost of each attraction, all in the same currency (use 0 for free entry).
        currency: The currency code of the costs, e.g. "NPR" or "USD".
    """
    print(f"\n[TOOL EXECUTED] calculate_total_cost(attractions={attractions}, costs={costs}, currency='{currency}')")
    tool_calls_log.append(("Calculator tool", f"calculate_total_cost({attractions}, {costs}, '{currency}')"))

    def failure(message: str) -> dict:
        print(f"    → ERROR: {message}")
        return {"ok": False, "error": message}

    # Check the input before calculating
    if len(attractions) != NUMBER_OF_ATTRACTIONS:
        return failure(f"Expected exactly {NUMBER_OF_ATTRACTIONS} attractions, got {len(attractions)}.")
    if len(costs) != len(attractions):
        return failure("Each attraction needs exactly one cost (same order as the names).")
    if any(cost < 0 for cost in costs):
        return failure("Costs cannot be negative.")

    total = round(sum(costs), 2)
    breakdown = [{"attraction": name, "cost": cost} for name, cost in zip(attractions, costs)]
    calculation = " + ".join(f"{name} ({cost:g})" for name, cost in zip(attractions, costs)) + f" = {total:g} {currency}"

    last_calculation.clear()
    last_calculation.update({"breakdown": breakdown, "currency": currency, "total": total, "calculation": calculation})

    print(f"    → {calculation}")
    return {"ok": True, "breakdown": breakdown, "currency": currency, "total": total, "calculation": calculation}

### Quick tool check (no LLM involved)

`@tool` functions can still be called like normal Python functions. This checks the weather and
calculator tools on their own, including one error case. (Search isn't tested here to save Tavily credits —
the agent run below uses it.)

In [25]:
print(get_current_weather(city="Kathmandu"))
print(get_current_weather(city="NotARealCity123"))          # should return an error, not crash
print(calculate_total_cost(
    attractions=["Boudhanath Stupa", "Pashupatinath Temple", "Swayambhunath"],
    costs=[400, 1000, 200],
    currency="NPR",
))
print(calculate_total_cost(attractions=["Only one place"], costs=[100], currency="NPR"))   # should be rejected

tool_calls_log.clear()
last_calculation.clear()


[TOOL EXECUTED] get_current_weather(city='Kathmandu')
    → Kathmandu, NP: 18.4 °C, moderate rain
{'ok': True, 'city': 'Kathmandu', 'country': 'NP', 'temperature_c': 18.4, 'feels_like_c': 18.6, 'humidity_percent': 88, 'description': 'moderate rain'}

[TOOL EXECUTED] get_current_weather(city='NotARealCity123')
    → ERROR: City not found by OpenWeather: 'NotARealCity123'.
{'ok': False, 'city': 'NotARealCity123', 'error': "City not found by OpenWeather: 'NotARealCity123'."}

[TOOL EXECUTED] calculate_total_cost(attractions=['Boudhanath Stupa', 'Pashupatinath Temple', 'Swayambhunath'], costs=[400, 1000, 200], currency='NPR')
    → Boudhanath Stupa (400) + Pashupatinath Temple (1000) + Swayambhunath (200) = 1600 NPR
{'ok': True, 'breakdown': [{'attraction': 'Boudhanath Stupa', 'cost': 400}, {'attraction': 'Pashupatinath Temple', 'cost': 1000}, {'attraction': 'Swayambhunath', 'cost': 200}], 'currency': 'NPR', 'total': 1600, 'calculation': 'Boudhanath Stupa (400) + Pashupatinath Temple (100

## 5. Strands agent configuration

The agent gets the three tools, the Groq model, and a **system prompt** with the rules.
`SequentialToolExecutor()` makes the tools run **one after another** (instead of in parallel),
as taught in the instructor's *MultiStep Tool Call* section — the calculator needs the prices
found by the search first.

In [ ]:
SYSTEM_PROMPT = """You are a travel planner that creates a concise one-day travel plan for a city.

Follow these steps, using the tools — never invent weather, attractions, prices or totals:
1. Call get_current_weather for the city.
2. Call search_web to find 3 popular tourist attractions in the city.
3. Call search_web to find the admission / entry fee of each attraction
   (you may search for several at once, e.g. "<city> attractions entry fee for foreigners").
   - Use prices exactly as found. Keep all costs in ONE currency (prefer the local currency).
   - If an attraction is free, use 0. If no price can be found, use 0 and clearly mark it
     as "price not found" in the plan.
4. 4. Call calculate_total_cost with the 3 attraction names and their costs (in the same order).
   Report the calculation and total EXACTLY as the tool returns them.
5. Write the final plan in this format (plain text, concise):

One-Day Travel Plan: <City>

Weather (weather tool):
<temperature> °C, <description>

Top 3 Attractions (web search):
1. <Name> — admission: <price or "free" or "price not found"> (source: <website name>)
2. ...
3. ...

Estimated Attraction Cost (calculator tool):
<calculation from the tool>

Itinerary:
<5–6 short lines with times, e.g. "08:00 — ...", visiting the 3 attractions, with a lunch break,
 and weather-appropriate advice>

Notes:
<one or two lines on assumptions, e.g. prices are for foreign visitors, prices may change>
"""


def create_travel_agent() -> Agent:
    """Creates a fresh agent (new conversation) with the three tools."""
    return Agent(
        model=groq_model,
        tools=[get_current_weather, search_web, calculate_total_cost],
        tool_executor=SequentialToolExecutor(),   # run tool calls one at a time
        system_prompt=SYSTEM_PROMPT,
        callback_handler=None,   # don't stream text while running; we print the final plan once below
    )


travel_agent = create_travel_agent()
print("Tools available to the agent:", travel_agent.tool_names)

Tools available to the agent: ['get_current_weather', 'search_web', 'calculate_total_cost']


## 6. Run the travel planner

Change `CITY` to plan a trip somewhere else. While the agent works, each tool prints a
`[TOOL EXECUTED]` line, so you can follow what it's doing.

In [27]:
CITY = "Kathmandu"

tool_calls_log.clear()
last_calculation.clear()
travel_agent = create_travel_agent()   # fresh agent, so a previous city doesn't affect this one

print(f"Planning a one-day trip to {CITY}...")
try:
    result = travel_agent(f"Create a one-day travel plan for {CITY}.")
    final_plan = str(result)
except Exception as exc:   # e.g. Groq rate limit or network problem
    final_plan = f"The travel planner could not finish: {type(exc).__name__}. Please try again in a minute."

Planning a one-day trip to Kathmandu...


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.



[TOOL EXECUTED] get_current_weather(city='Kathmandu')
    → Kathmandu, NP: 18.4 °C, moderate rain

[TOOL EXECUTED] search_web(query='top tourist attractions Kathmandu')


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


    → 3 result(s): 50+ Top Nepal Tourist Attractions, Places to Visit | Kathmandu: 15 BEST Tours and Things to Do in 2026 | 50+ Places To Visit in Kathmandu: Museums, Temples

[TOOL EXECUTED] search_web(query='Pashupatinath Temple entry fee foreign visitor')


...

    → 3 result(s): Boudhanath Stupa : Origin, Significance, Location, | Nepal Attractions: Fees & Timings | Heritage Site Entry Fees

[TOOL EXECUTED] search_web(query='Swayambhunath entry fee foreign')


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.


    → 3 result(s): Heritage Site Entry Fees | Complete Guide to Entrance Fees in Nepal: Temples  | Heritage Sites and Monuments entrance fees in Nepa


reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.
reasoningContent is not supported in multi-turn conversations with the Chat Completions API.



[TOOL EXECUTED] calculate_total_cost(attractions=['Pashupatinath Temple', 'Boudhanath Stupa', 'Swayambhunath Stupa'], costs=[1000.0, 400.0, 200.0], currency='NPR')
    → Pashupatinath Temple (1000) + Boudhanath Stupa (400) + Swayambhunath Stupa (200) = 1600 NPR


## 7. Final itinerary and tool-call check

- **Tool calls** — what the agent actually did (from `tool_calls_log`).
- **Calculator result** — the authoritative total, straight from the calculator tool.
- **Final plan** — the itinerary written by the model from the tool results.

In [28]:
print("=" * 60)
print("TOOL CALLS MADE BY THE AGENT (in order)")
print("=" * 60)
for number, (tool_type, call) in enumerate(tool_calls_log, start=1):
    print(f"{number}. {tool_type:<16} {call}")

used = {tool_type for tool_type, _ in tool_calls_log}
for required in ["Weather tool", "Search tool", "Calculator tool"]:
    print(f"   {'✓' if required in used else '✗ NOT USED:'} {required}")

print("\n" + "=" * 60)
print("CALCULATOR RESULT (authoritative total)")
print("=" * 60)
if last_calculation:
    print(last_calculation["calculation"])
else:
    print("The calculator tool was not called, so there is no verified total.")

print("\n" + "=" * 60)
print("FINAL PLAN (written by the agent from the tool results)")
print("=" * 60)
print(final_plan)

TOOL CALLS MADE BY THE AGENT (in order)
1. Weather tool     get_current_weather('Kathmandu')
2. Search tool      search_web('top tourist attractions Kathmandu')
3. Search tool      search_web('Pashupatinath Temple entry fee foreign visitor')
4. Search tool      search_web('Boudhanath Stupa entry fee foreign')
5. Search tool      search_web('Swayambhunath entry fee foreign')
6. Calculator tool  calculate_total_cost(['Pashupatinath Temple', 'Boudhanath Stupa', 'Swayambhunath Stupa'], [1000.0, 400.0, 200.0], 'NPR')
   ✓ Weather tool
   ✓ Search tool
   ✓ Calculator tool

CALCULATOR RESULT (authoritative total)
Pashupatinath Temple (1000) + Boudhanath Stupa (400) + Swayambhunath Stupa (200) = 1600 NPR

FINAL PLAN (written by the agent from the tool results)
One-Day Travel Plan: Kathmandu  

Weather (weather tool):  
18.4 °C, moderate rain  

Top 3 Attractions (web search):  
1. Pashupatinath Temple — admission: NPR 1,000 (source: Nepal Planet Treks)  
2. Boudhanath Stupa — admission: NPR 4

## 8. Error handling and notes

**Error handling**
- **Weather tool:** unknown city, timeout, bad key or unexpected data → returns `{"ok": False, "error": ...}` so the agent can report it instead of crashing. The key is never shown (the raw request URL contains it).
- **Search tool:** any Tavily failure (network, key, usage limit) → returns a short error message.
- **Calculator tool:** requires exactly 3 attractions with one cost each, and rejects negative costs.
- **Agent run:** wrapped in `try/except`, so a Groq rate limit or network error prints a friendly message.

**Notes and assumptions**
- Attractions and prices come from live web search, so results can vary between runs, and prices may be outdated on the source websites.
- Prices are kept in one currency (usually the local one). Where no price is found, the plan says so and it counts as 0 in the total.
- The total is always computed by `calculate_total_cost`, and Section 7 prints that exact result so it can be compared with the plan.